In [2]:
import os
import cv2
import time
import numpy as np
from collections import deque
import mediapipe as mp
from sklearn.metrics import classification_report, confusion_matrix

# --------------------------
# Your folders & gestures
# --------------------------
VIDEO_DIR = "C:/Users/Samrudhi/Videos/Samrudhi/3rdSem/AI Lab/gesture_videos"
GESTURES = ["jump", "duck", "left", "right", "neutral"]

# Small helper to store predictions
y_true = []
y_pred = []

# --------------------------
# Mediapipe setup
# --------------------------
mp_holistic = mp.solutions.holistic

def landmark_xy(landmarks, idx):
    lm = landmarks[idx]
    return lm.x, lm.y, lm.visibility

def center(a, b):
    return ((a[0] + b[0]) * 0.5, (a[1] + b[1]) * 0.5)

def visible(*vs, thr=0.5):
    return all(v >= thr for v in vs)

# --------------------------
# Detection logic (same as game)
# --------------------------
LEAN_THRESH = 0.06
HANDS_ABOVE_SHOULDERS_DELTA = 0.02
CROUCH_TORSO_RATIO = 0.62

def detect_gesture(pose, ema_state):
    """
    Returns: 'jump','duck','left','right','neutral'
    """

    if not pose:
        return "neutral"

    lms = pose.landmark

    NOSE = mp_holistic.PoseLandmark.NOSE
    L_SHO = mp_holistic.PoseLandmark.LEFT_SHOULDER
    R_SHO = mp_holistic.PoseLandmark.RIGHT_SHOULDER
    L_HIP = mp_holistic.PoseLandmark.LEFT_HIP
    R_HIP = mp_holistic.PoseLandmark.RIGHT_HIP
    L_WRIST = mp_holistic.PoseLandmark.LEFT_WRIST
    R_WRIST = mp_holistic.PoseLandmark.RIGHT_WRIST

    nose = landmark_xy(lms, NOSE)
    lsho = landmark_xy(lms, L_SHO)
    rsho = landmark_xy(lms, R_SHO)
    lhip = landmark_xy(lms, L_HIP)
    rhip = landmark_xy(lms, R_HIP)
    lwri = landmark_xy(lms, L_WRIST)
    rwri = landmark_xy(lms, R_WRIST)

    if not visible(nose[2], lsho[2], rsho[2], lhip[2], rhip[2], lwri[2], rwri[2]):
        return "neutral"

    sh_cx, sh_cy = center(lsho, rsho)
    hip_cx, hip_cy = center(lhip, rhip)

    dx = sh_cx - hip_cx
    torso = abs(nose[1] - hip_cy)

    # Wrist positions relative to shoulders
    wrist_left_y = lwri[1]
    wrist_right_y = rwri[1]
    shoulder_y = (lsho[1] + rsho[1]) * 0.5

    # JUMP
    if wrist_left_y < (shoulder_y - HANDS_ABOVE_SHOULDERS_DELTA) and \
       wrist_right_y < (shoulder_y - HANDS_ABOVE_SHOULDERS_DELTA):
        return "jump"

    # DUCK
    if torso < CROUCH_TORSO_RATIO:
        return "duck"

    # LEFT / RIGHT
    if dx <= -LEAN_THRESH:
        return "left"
    if dx >= LEAN_THRESH:
        return "right"

    # DEFAULT
    return "neutral"

# --------------------------
# Main evaluation loop
# --------------------------
def evaluate_videos():
    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        enable_segmentation=False,
        refine_face_landmarks=False,
        smooth_landmarks=True
    ) as holistic:

        for gesture in GESTURES:
            print(f"\n=== Evaluating: {gesture} ===")

            for file in os.listdir(VIDEO_DIR):
                if not file.startswith(gesture):
                    continue

                filepath = os.path.join(VIDEO_DIR, file)
                cap = cv2.VideoCapture(filepath)

                frame_predictions = []

                while True:
                    ok, frame = cap.read()
                    if not ok:
                        break

                    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    result = holistic.process(rgb)
                    pose = result.pose_landmarks

                    pred = detect_gesture(pose, None)
                    frame_predictions.append(pred)

                cap.release()

                # Majority vote for final prediction
                final_pred = max(set(frame_predictions), key=frame_predictions.count)

                y_true.append(gesture)
                y_pred.append(final_pred)

                print(f"{file} → predicted: {final_pred}")

    # --------------------------
    # Print metrics
    # --------------------------
    print("\n\n===== CLASSIFICATION REPORT =====")
    print(classification_report(y_true, y_pred, digits=3))

    print("\n===== CONFUSION MATRIX =====")
    print(confusion_matrix(y_true, y_pred, labels=GESTURES))


if __name__ == "__main__":
    evaluate_videos()



=== Evaluating: jump ===
jump_1.mp4 → predicted: jump
jump_2.mp4 → predicted: jump
jump_3.mp4 → predicted: jump

=== Evaluating: duck ===
duck_1.mp4 → predicted: duck
duck_2.mp4 → predicted: duck
duck_3.mp4 → predicted: duck

=== Evaluating: left ===
left_1.mp4 → predicted: duck
left_2.mp4 → predicted: duck
left_3.mp4 → predicted: duck

=== Evaluating: right ===
right_1.mp4 → predicted: duck
right_2.mp4 → predicted: duck
right_3.mp4 → predicted: duck

=== Evaluating: neutral ===
neutral_1.mp4 → predicted: duck
neutral_2.mp4 → predicted: duck
neutral_3.mp4 → predicted: duck


===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

        duck      0.250     1.000     0.400         3
        jump      1.000     1.000     1.000         3
        left      0.000     0.000     0.000         3
     neutral      0.000     0.000     0.000         3
       right      0.000     0.000     0.000         3

    accuracy                          0.400        15
   

c:\Users\Samrudhi\anaconda3\envs\gesturegame\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Samrudhi\anaconda3\envs\gesturegame\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Samrudhi\anaconda3\envs\gesturegame\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi